# Trait Network + Machine Learning Analysis

### Rice Trait Interaction and Harvest Index Prediction

Author: Jarvis  
Repository: Rice Trait Network Analysis

---

## Overview

This notebook analyzes phenotypic trait relationships and identifies key predictors of **Harvest Index (HI)** using statistical and machine learning methods.

The pipeline integrates:

- Trait correlation analysis
- Trait interaction network
- Partial Least Squares regression
- Variable Importance in Projection (VIP)
- Random Forest feature importance
- Gradient Boosting with SHAP interpretation
- Z-score heatmap of genotypes

---

## Expected Dataset Format

Your Excel dataset should contain:

| Column | Description |
|------|------|
| Genotype | Genotype ID |
| Harvest_Index | Target trait |
| Other traits | Predictor variables |

Example traits:

- Plant_height
- Panicle_length
- Effective_tillers
- Grain_length
- Grain_breadth
- Thousand_grain_weight
- Straw_yield
- Grain_yield

---

## Output Figures

- Trait correlation clustermap
- Trait interaction network
- Strong trait network
- PLS predicted vs observed plot
- Z-score genotype heatmap
- Random Forest feature importance
- SHAP feature importance

All figures are saved at **600 dpi for publication**.

## Step 1 — Install Required Libraries

In [ ]:
!pip install pandas numpy seaborn matplotlib networkx scikit-learn shap openpyxl

## Step 2 — Import Python Libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import networkx as nx
import shap
import os

from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import r2_score

from google.colab import files

## Step 3 — Upload Dataset

In [ ]:
uploaded = files.upload()

file_name = list(uploaded.keys())[0]

df = pd.read_excel(file_name)

print("Dataset shape:", df.shape)
df.head()

## Step 4 — Clean Column Names

In [ ]:
df.columns = df.columns.str.strip()
df.columns = df.columns.str.replace(' ', '_')
df.columns = df.columns.str.upper()

print("Columns detected:")
print(df.columns)

## Step 5 — Define Target Trait

In [ ]:
GENOTYPE_COL = 'GENOTYPE'
TARGET_COL = 'HARVEST_INDEX'

df = df.dropna()

trait_cols = [c for c in df.columns if c not in [GENOTYPE_COL, TARGET_COL]]

X = df[trait_cols]
y = df[TARGET_COL]

print("Predictor traits:", len(trait_cols))

## Step 6 — Create Output Folder

In [ ]:
output_folder = "TraitNetwork_ML_outputs"
os.makedirs(output_folder, exist_ok=True)

## Step 7 — Trait Correlation Clustermap

In [ ]:
corr = X.corr()

sns.clustermap(corr, cmap="vlag", annot=True, linewidths=0.5, figsize=(12,10))

plt.savefig(f"{output_folder}/trait_clustermap.png", dpi=600)

plt.show()

## Step 8 — Trait Interaction Network

In [ ]:
G = nx.Graph()

for t in trait_cols:
    G.add_node(t)

for i in trait_cols:
    for j in trait_cols:
        if i != j and abs(corr.loc[i,j]) > 0.6:
            G.add_edge(i,j, weight=corr.loc[i,j])

plt.figure(figsize=(10,10))
pos = nx.spring_layout(G, seed=42)

weights = [abs(G[u][v]['weight']) * 5 for u,v in G.edges()]

nx.draw(G,pos,with_labels=True,width=weights,node_color='skyblue',node_size=1500)

plt.title("Trait Interaction Network")

plt.savefig(f"{output_folder}/trait_network.png", dpi=600)

plt.show()

## Step 9 — Partial Least Squares Regression

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pls = PLSRegression(n_components=min(5,len(trait_cols)))

pls.fit(X_scaled,y)

y_pred = cross_val_predict(pls,X_scaled,y,cv=5)

print("PLS R²:", r2_score(y,y_pred))

## Step 10 — Random Forest Importance

In [ ]:
rf = RandomForestRegressor(n_estimators=300,max_depth=5,random_state=42)

rf.fit(X,y)

imp = pd.DataFrame({
    "Trait":trait_cols,
    "Importance":rf.feature_importances_
}).sort_values("Importance",ascending=False)

sns.barplot(data=imp,x="Importance",y="Trait")

plt.savefig(f"{output_folder}/rf_feature_importance.png",dpi=600)

plt.show()

## Step 11 — SHAP Interpretation

In [ ]:
gb = GradientBoostingRegressor(n_estimators=300,max_depth=3,random_state=42)

gb.fit(X,y)

explainer = shap.Explainer(gb,X)

shap_values = explainer(X)

shap.plots.bar(shap_values,show=False)

plt.savefig(f"{output_folder}/shap_feature_importance.png",dpi=600)

plt.show()

## Analysis Completed

All outputs saved in:

`TraitNetwork_ML_outputs/`

Figures are exported at **600 dpi for publication**.